[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/21.%20The%20Yard%20Block%20Housekeeping%20Problem/P21-Tier-7_executed.ipynb)

---


# 21. The Yard Block Housekeeping Problem

## Tier 7: Human–AI Symbiotic Partnership

### Goal and scope

This notebook demonstrates a structured human–AI decision workflow for yard-block housekeeping. A rule-based AI component analyzes a simplified yard state and operational context, produces a recommendation with an explanation, and presents it for human review. The operator can accept, modify, request an alternative, or reject the recommendation. The interaction is recorded as feedback for later analysis.

The code is an instructional simulation, not a validated terminal planning system. It does not optimize a mathematical objective, execute physical moves, train a machine-learning model, or establish that collaboration outperforms either human-only or AI-only decisions. Reported confidence, costs, completion estimates, and collaboration-quality scores are illustrative quantities generated by explicit rules; they are not calibrated performance measurements.

### Inputs, outputs, and assumptions

**Inputs:** block loads and capacities; weather, equipment, staffing, vessel, and gate information; and customer priorities. The demonstration creates these inputs in Python.

**Outputs:** an AI recommendation, a human evaluation, a final decision record, captured feedback, and descriptive scenario plots.

**Model assumptions:**
- Blocks numbered 1 and 2 are treated as locations containing premium-customer containers. This is a demonstration convention, not a property inferred from container-level data.
- A proposed move transfers a quantity from an urgent block to an available block. The rule-based recommender limits the quantity per move, but does not model individual containers, stacking, travel distances, equipment routing, or detailed feasibility constraints.
- Weather, staffing, equipment status, and customer information influence review rules; the notebook does not calculate a validated operational cost or safety risk from them.
- A captured human insight is stored as a record. It is not automatically incorporated into the AI recommendation rules.

### Collaboration workflow

The teaching sequence is: (1) represent the operational context and feedback, (2) generate a rule-based recommendation and explanation, (3) evaluate it with a human operator, (4) form a final decision, and (5) record and summarize the interaction. The stress scenario and plots examine how this illustrative workflow responds to different inputs; they do not establish statistical performance.

### Formal view of the workflow

Let \(x\) denote the supplied yard state, \(c\) the operational context, and \(q\) the customer constraints. The AI component applies a deterministic set of rules \(R\) to produce a recommendation \(a\) and explanation \(e\):

$$
(a,e)=R(x,c,q).
$$

The human-review rules \(H\) use the recommendation, explanation, context, and customer constraints to produce feedback \(h\):

$$
h=H(a,e,c,q).
$$

The collaboration rule \(D\) combines \(a\), \(h\), and \(e\) into the final decision \(d\), after which the learning-capture routine stores records associated with the interaction:

$$
d=D(a,h,e), \qquad \ell=L(a,h,d).
$$

Here, \(R\), \(H\), \(D\), and \(L\) are implemented by `ExplainableAI`, `HumanOperator`, and `HumanAICollaboration`. These are rule-based functions, not an optimization model or a learned policy. Their outputs depend on the supplied data and the stated rules. The notebook contains no claim that a resulting decision is globally optimal.

### Interpreting results

A recommendation is a proposal for human review, not an instruction to execute. A confidence score is a heuristic score defined by the code, not a probability that the decision is correct. The scenario comparison describes outputs under two generated inputs; it cannot by itself show that one approach is better or that results generalize to real operations.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
import copy

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class DecisionConfidence(Enum):
    """Qualitative labels available for describing decision confidence."""

    VERY_LOW = "very_low"
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    VERY_HIGH = "very_high"


class HumanFeedbackType(Enum):
    """Feedback categories used in the collaboration workflow."""

    ACCEPT = "accept"
    MODIFY = "modify"
    REJECT = "reject"
    REQUEST_ALTERNATIVES = "request_alternatives"
    PROVIDE_CONTEXT = "provide_context"


@dataclass
class AIExplanation:
    """Reasoning, uncertainty, and outcome fields shown to the operator."""

    primary_reasoning: str
    secondary_reasoning: str
    confidence_level: float
    alternatives: List[Dict[str, Any]]
    risks: List[str]
    data_sources: List[str]
    uncertainty_factors: List[str]
    expected_outcomes: Dict[str, float]


@dataclass
class HumanInsight:
    """A human observation recorded as feedback for later review."""

    insight_type: str
    context: str
    reasoning: str
    confidence: float
    priority: float
    applicable_conditions: List[str]
    learned_patterns: List[str]


@dataclass
class CollaborativeDecision:
    """Complete record of one human–AI decision interaction."""

    timestamp: datetime
    ai_recommendation: Dict[str, Any]
    ai_explanation: AIExplanation
    human_feedback: Optional[Dict[str, Any]]
    human_insights: List[HumanInsight]
    final_decision: Dict[str, Any]
    collaboration_quality: float
    learning_outcomes: List[str]


@dataclass
class CustomerConstraint:
    """Customer preferences and service information used by review rules."""

    customer_id: str
    customer_name: str
    priority_level: str
    handling_requirements: List[str]
    service_level_agreement: Dict[str, Any]
    special_instructions: List[str]
    cost_sensitivity: float
    time_sensitivity: float


@dataclass
class OperationalContext:
    """Operational information available to the AI and human review rules."""

    timestamp: datetime
    weather_conditions: Dict[str, Any]
    equipment_status: Dict[str, str]
    staff_availability: Dict[str, int]
    vessel_schedule: List[Dict[str, Any]]
    gate_traffic_level: str
    maintenance_activities: List[str]
    safety_constraints: List[str]
    regulatory_requirements: List[str]


@dataclass
class LearningCapture:
    """Stored interaction record; storage does not train or update the AI."""

    interaction_id: str
    timestamp: datetime
    situation_description: str
    human_modification: Dict[str, Any]
    original_ai_recommendation: Dict[str, Any]
    outcome_better: Optional[bool]
    learning_extracted: str
    generalizable_pattern: bool
    confidence_improvement: float


@dataclass
class YardBlock:
    """Aggregate block data used by the illustrative scenario rules."""

    id: int
    current_load: int
    capacity: int
    utilization: float
    available_capacity: int

In [ ]:
class ExplainableAI:
    """Rule-based housekeeping recommender with a human-readable explanation."""

    def __init__(self):
        self.decision_history = []
        self.pattern_library = {}
        self.explanation_templates = {
            "high_priority_vessel": {
                "primary": "Preparing for a high-priority vessel arrival",
                "secondary": "Reviewing block moves in light of vessel timing",
                "risks": ["Weather delay", "Equipment failure", "Gate congestion"],
            },
            "weather_constraint": {
                "primary": "Accounting for adverse weather conditions",
                "secondary": "Prioritizing operational safety and continuity",
                "risks": ["Reduced equipment capacity", "Longer operation times", "Safety concerns"],
            },
            "equipment_optimization": {
                "primary": "Reviewing equipment availability",
                "secondary": "Considering equipment use across the proposed moves",
                "risks": ["Equipment bottlenecks", "Maintenance conflicts", "Operator fatigue"],
            },
            "customer_service": {
                "primary": "Considering customer service requirements",
                "secondary": "Reviewing potential effects on priority customers",
                "risks": ["Customer dissatisfaction", "Service-level violations", "Revenue impact"],
            },
        }

    def generate_recommendation(
        self,
        yard_state: Dict[str, Any],
        context: OperationalContext,
        customer_constraints: List[CustomerConstraint],
    ) -> Tuple[Dict[str, Any], AIExplanation]:
        """Apply the recommendation rules and create an explanation."""
        analysis = self._analyze_situation(yard_state, context, customer_constraints)
        recommendation = self._generate_base_recommendation(yard_state, analysis)
        explanation = self._create_explanation(recommendation, analysis, context)
        self.decision_history.append(recommendation)
        return recommendation, explanation

    def _analyze_situation(
        self,
        yard_state: Dict[str, Any],
        context: OperationalContext,
        customer_constraints: List[CustomerConstraint],
    ) -> Dict[str, Any]:
        """Classify blocks and operational signals for the subsequent rules."""
        analysis = {
            "urgent_blocks": [],
            "available_blocks": [],
            "equipment_constraints": [],
            "customer_priorities": [],
            "weather_impact": 1.0,
            "time_pressure": "normal",
            "risk_factors": [],
            "opportunities": [],
        }

        for block_id, block in yard_state.get("blocks", {}).items():
            if block.utilization > 0.9:
                analysis["urgent_blocks"].append(block_id)
            elif block.utilization < 0.7:
                analysis["available_blocks"].append(block_id)

        analysis["equipment_constraints"] = [
            equipment_id
            for equipment_id, status in context.equipment_status.items()
            if status != "operational"
        ]
        analysis["customer_priorities"] = [
            constraint.customer_id
            for constraint in customer_constraints
            if constraint.priority_level == "premium"
        ]

        weather = context.weather_conditions
        if weather.get("wind_speed", 0) > 30:
            analysis["weather_impact"] = 0.6
            analysis["risk_factors"].append("high_wind")
        elif weather.get("precipitation", 0) > 5:
            analysis["weather_impact"] = 0.8
            analysis["risk_factors"].append("heavy_rain")

        upcoming_vessels = [
            vessel
            for vessel in context.vessel_schedule
            if vessel.get("eta", context.timestamp)
            <= context.timestamp + timedelta(hours=4)
        ]
        if len(upcoming_vessels) >= 2:
            analysis["time_pressure"] = "high"
            analysis["risk_factors"].append("vessel_congestion")
        elif upcoming_vessels:
            analysis["time_pressure"] = "medium"

        if context.gate_traffic_level == "low":
            analysis["opportunities"].append("low_gate_traffic")
        if not analysis["equipment_constraints"]:
            analysis["opportunities"].append("full_equipment_availability")
        return analysis

    def _generate_base_recommendation(
        self, yard_state: Dict[str, Any], analysis: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Create a bounded move list using the notebook's stated heuristic rules."""
        recommendation = {
            "moves": [],
            "time_window": "3_hours",
            "equipment_allocation": {},
            "priority_score": 0.0,
            "estimated_cost": 0.0,
            "expected_completion": 0.0,
            "risk_level": "medium",
        }

        for urgent_block in analysis["urgent_blocks"][:3]:
            for available_block in analysis["available_blocks"][:2]:
                if len(recommendation["moves"]) >= 10:
                    break
                source = yard_state["blocks"][urgent_block]
                destination = yard_state["blocks"][available_block]
                quantity = min(
                    20,
                    int(source.current_load * 0.1),
                    max(0, destination.available_capacity // 2),
                )
                if quantity > 0:
                    move = {
                        "source_block": urgent_block,
                        "destination_block": available_block,
                        "quantity": quantity,
                        "priority": "high",
                        "estimated_duration": quantity * 2,
                        "cost": quantity * 15,
                    }
                    recommendation["moves"].append(move)
                    recommendation["estimated_cost"] += move["cost"]

        urgency_factor = len(analysis["urgent_blocks"]) * 0.3
        time_pressure_factor = {"high": 0.4, "medium": 0.2, "normal": 0.0}[
            analysis["time_pressure"]
        ]
        weather_factor = (1.0 - analysis["weather_impact"]) * 0.2
        customer_factor = len(analysis["customer_priorities"]) * 0.1
        recommendation["priority_score"] = (
            urgency_factor + time_pressure_factor + weather_factor + customer_factor
        )

        if len(analysis["risk_factors"]) >= 3 or analysis["weather_impact"] < 0.7:
            recommendation["risk_level"] = "high"
        elif analysis["risk_factors"]:
            recommendation["risk_level"] = "medium"
        else:
            recommendation["risk_level"] = "low"

        total_urgent_excess = sum(
            max(
                0,
                yard_state["blocks"][block_id].current_load
                - int(yard_state["blocks"][block_id].capacity * 0.8),
            )
            for block_id in analysis["urgent_blocks"]
        )
        total_moved = sum(move["quantity"] for move in recommendation["moves"])
        recommendation["expected_completion"] = min(
            1.0, total_moved / max(1, total_urgent_excess)
        )
        return recommendation

    def _create_explanation(
        self,
        recommendation: Dict[str, Any],
        analysis: Dict[str, Any],
        context: OperationalContext,
    ) -> AIExplanation:
        """Build an explanation whose scores are explicitly heuristic."""
        if analysis["time_pressure"] == "high":
            template_key = "high_priority_vessel"
        elif analysis["weather_impact"] < 0.8:
            template_key = "weather_constraint"
        elif analysis["customer_priorities"]:
            template_key = "customer_service"
        else:
            template_key = "equipment_optimization"

        template = self.explanation_templates[template_key]
        primary = template["primary"]
        secondary = template["secondary"]
        if analysis["urgent_blocks"]:
            primary += " (high-utilization blocks: " + ", ".join(
                map(str, analysis["urgent_blocks"][:3])
            ) + ")"
        if analysis["time_pressure"] != "normal":
            secondary += f" ({analysis['time_pressure']} time pressure)"

        confidence = 0.8 - 0.1 * len(analysis["risk_factors"])
        if analysis["weather_impact"] < 0.8:
            confidence -= 0.2
        confidence += 0.1 * len(analysis["opportunities"])
        confidence = max(0.3, min(0.95, confidence))

        moves = recommendation["moves"]
        half_moves = moves[: len(moves) // 2]
        alternatives = [
            {
                "description": "Reduced scope: focus on a subset of proposed moves",
                "moves": half_moves,
                "completion": recommendation["expected_completion"] * 0.6,
                "cost": sum(move["cost"] for move in half_moves),
                "duration": sum(move["estimated_duration"] for move in half_moves),
                "time_window": "3_hours",
            },
            {
                "description": "Extended timeline: spread the proposed moves over four hours",
                "moves": moves.copy(),
                "completion": min(1.0, recommendation["expected_completion"] * 1.1),
                "cost": recommendation["estimated_cost"] * 0.9,
                "duration": sum(move["estimated_duration"] for move in moves) * 1.3,
                "time_window": "4_hours",
            },
        ]
        if analysis["weather_impact"] < 0.8:
            weather_moves = moves[: len(moves) // 3]
            alternatives.append(
                {
                    "description": "Weather-adaptive: defer non-critical moves",
                    "moves": weather_moves,
                    "completion": recommendation["expected_completion"] * 0.4,
                    "cost": sum(move["cost"] for move in weather_moves),
                    "duration": sum(move["estimated_duration"] for move in weather_moves),
                    "time_window": "3_hours",
                }
            )

        outcomes = {
            "vessel_delay_reduction": recommendation["priority_score"] * 10,
            "yard_utilization_improvement": recommendation["expected_completion"] * 15,
            "equipment_efficiency": len(moves) * 2,
            "cost_per_move": recommendation["estimated_cost"] / max(1, len(moves)),
        }
        return AIExplanation(
            primary_reasoning=primary,
            secondary_reasoning=secondary,
            confidence_level=confidence,
            alternatives=alternatives,
            risks=template["risks"] + analysis["risk_factors"],
            data_sources=["yard_state", "equipment_status", "weather_conditions", "vessel_schedule"],
            uncertainty_factors=["demand_fluctuations", "equipment_reliability", "forecast_accuracy"],
            expected_outcomes=outcomes,
        )

In [ ]:
class HumanOperator:
    """Apply explicit human-review rules to a recommendation."""

    def __init__(self, name: str, experience_years: int, expertise_areas: List[str]):
        self.name = name
        self.experience_years = experience_years
        self.expertise_areas = expertise_areas
        self.decision_history = []
        self.insights_database = []
        self.confidence_calibration = 0.85
        self.risk_tolerance = 0.6
        self.customer_sensitivity = 0.8
        self.safety_consciousness = 0.9
        self.cost_sensitivity = 0.7

    def evaluate_ai_recommendation(
        self,
        recommendation: Dict[str, Any],
        explanation: AIExplanation,
        context: OperationalContext,
        customer_constraints: List[CustomerConstraint],
    ) -> Dict[str, Any]:
        """Return feedback and reasons based on the human-review rules."""
        customer_issues = self._check_customer_constraints(recommendation, customer_constraints)
        safety_issues = self._check_safety_concerns(recommendation, context)
        practicality_issues = self._check_operational_practicality(recommendation, context)
        concerns = customer_issues + safety_issues + practicality_issues
        evaluation = {
            "feedback_type": "",
            "modifications": {},
            "reasoning": "",
            "confidence": 0.0,
            "insights": [],
            "concerns": concerns,
            "acceptance_threshold": 0.7,
        }

        if not concerns and explanation.confidence_level >= evaluation["acceptance_threshold"]:
            evaluation["feedback_type"] = "accept"
            evaluation["confidence"] = explanation.confidence_level
            evaluation["reasoning"] = "The recommendation passes the modeled review checks."
        elif len(concerns) <= 2 and explanation.confidence_level >= 0.6:
            evaluation["feedback_type"] = "modify"
            evaluation["modifications"] = self._generate_modifications(recommendation, concerns)
            evaluation["confidence"] = explanation.confidence_level * 0.9
            evaluation["reasoning"] = "Modified to address: " + "; ".join(concerns)
            evaluation["insights"] = self._extract_insights(concerns, context)
        elif explanation.alternatives:
            evaluation["feedback_type"] = "request_alternatives"
            evaluation["confidence"] = 0.5
            evaluation["reasoning"] = "The recommendation has multiple concerns; review an alternative."
        else:
            evaluation["feedback_type"] = "reject"
            evaluation["confidence"] = 0.3
            evaluation["reasoning"] = "The recommendation was rejected because of the identified concerns."

        self.decision_history.append(evaluation)
        return evaluation

    def _check_customer_constraints(
        self,
        recommendation: Dict[str, Any],
        customer_constraints: List[CustomerConstraint],
    ) -> List[str]:
        """Apply the demonstration's premium-block and time-sensitivity checks."""
        concerns = []
        for constraint in customer_constraints:
            if constraint.priority_level == "premium" and any(
                move["source_block"] in [1, 2] for move in recommendation["moves"]
            ):
                concerns.append(
                    f"Premium customer {constraint.customer_name} containers may be affected"
                )
            if constraint.time_sensitivity > 0.8 and recommendation["time_window"] == "3_hours":
                concerns.append(
                    f"Time-sensitive customer {constraint.customer_name} may experience delays"
                )
        return concerns

    def _check_safety_concerns(
        self, recommendation: Dict[str, Any], context: OperationalContext
    ) -> List[str]:
        """Flag weather, equipment, and staffing conditions in the scenario."""
        concerns = []
        weather = context.weather_conditions
        if weather.get("wind_speed", 0) > 25 and recommendation["time_window"] == "3_hours":
            concerns.append("High wind conditions warrant a review of the operation window")
        if weather.get("precipitation", 0) > 10:
            concerns.append("Heavy rain warrants a review of equipment-operation safety")
        allocated = recommendation.get("equipment_allocation", {})
        if len(context.equipment_status) < len(allocated):
            concerns.append("The modeled equipment allocation exceeds listed equipment")
        total_staff = sum(context.staff_availability.values())
        if total_staff < 10 and len(recommendation["moves"]) > 5:
            concerns.append("Limited staff availability may constrain extensive operations")
        return concerns

    def _check_operational_practicality(
        self, recommendation: Dict[str, Any], context: OperationalContext
    ) -> List[str]:
        """Check the scenario's gate, maintenance, and duration thresholds."""
        concerns = []
        move_count = len(recommendation["moves"])
        if context.gate_traffic_level == "high" and move_count > 8:
            concerns.append("High gate traffic may interfere with housekeeping operations")
        if context.maintenance_activities and move_count > 6:
            concerns.append("Maintenance activities may limit equipment availability")
        duration = sum(move["estimated_duration"] for move in recommendation["moves"])
        if recommendation["time_window"] == "3_hours" and duration > 150:
            concerns.append("Estimated move duration exceeds the modeled 150-minute threshold")
        return concerns

    def _generate_modifications(
        self, recommendation: Dict[str, Any], concerns: List[str]
    ) -> Dict[str, Any]:
        """Apply the notebook's simple scope and timing modifications."""
        modifications = copy.deepcopy(recommendation)
        if any("premium customer" in concern.lower() for concern in concerns):
            modifications["moves"] = [
                move for move in modifications["moves"] if move["source_block"] not in [1, 2]
            ]
            modifications["reasoning"] = "Excluded moves from the designated premium blocks."
        if any("weather" in concern.lower() or "wind" in concern.lower() for concern in concerns):
            modifications["time_window"] = "2_hours"
            modifications["moves"] = modifications["moves"][: len(modifications["moves"]) // 2]
            modifications["reasoning"] = "Reduced the proposed scope in response to weather concerns."
        if any("gate traffic" in concern.lower() for concern in concerns):
            modifications["reasoning"] = "Timing requires review in light of gate traffic."
        modifications["estimated_cost"] = sum(move["cost"] for move in modifications["moves"])
        modifications["priority_score"] *= 0.9
        modifications["expected_completion"] *= 0.8
        return modifications

    def _extract_insights(
        self, concerns: List[str], context: OperationalContext
    ) -> List[HumanInsight]:
        """Convert selected concerns into records for later inspection."""
        insights = []
        if any("weather" in concern.lower() or "wind" in concern.lower() for concern in concerns):
            insights.append(
                HumanInsight(
                    insight_type="weather_adaptation",
                    context="Weather uncertainty in terminal operations",
                    reasoning="Review forecast uncertainty and maintain appropriate safety buffers.",
                    confidence=0.9,
                    priority=0.8,
                    applicable_conditions=["high_wind_forecast", "changing_weather_patterns"],
                    learned_patterns=["review_weather_buffer", "prioritize_safety"],
                )
            )
        if any("customer" in concern.lower() for concern in concerns):
            insights.append(
                HumanInsight(
                    insight_type="customer_relationship",
                    context="Premium customer service requirements",
                    reasoning="Review whether priority-customer containers should be excluded from housekeeping moves.",
                    confidence=0.95,
                    priority=0.9,
                    applicable_conditions=["premium_customers", "service_level_agreements"],
                    learned_patterns=["review_premium_container_moves", "consider_customer_service"],
                )
            )
        self.insights_database.extend(insights)
        return insights

In [ ]:
class HumanAICollaboration:
    """Coordinate recommendation, review, decision, and record keeping."""

    def __init__(self):
        self.explainable_ai = ExplainableAI()
        self.human_operator = HumanOperator(
            name="Sarah Chen",
            experience_years=12,
            expertise_areas=["yard_optimization", "customer_service", "safety_management"],
        )
        self.collaboration_history = []
        self.learning_database = []
        self.performance_metrics = {
            "ai_only": [],
            "human_only": [],
            "collaborative": [],
            "improvement_over_ai": [],
            "improvement_over_human": [],
            "satisfaction_scores": [],
            "learning_captured": [],
        }

    def simulate_collaborative_session(
        self,
        yard_state: Dict[str, Any],
        context: OperationalContext,
        customer_constraints: List[CustomerConstraint],
    ) -> CollaborativeDecision:
        """Run one complete recommendation and human-review cycle."""
        print("\n" + "=" * 80)
        print("HUMAN-AI COLLABORATIVE HOUSEKEEPING SESSION")
        print("=" * 80)
        print("\n=== STEP 1: AI RECOMMENDATION ===")
        recommendation, explanation = self.explainable_ai.generate_recommendation(
            yard_state, context, customer_constraints
        )
        self._display_ai_recommendation(recommendation, explanation)

        print("\n=== STEP 2: HUMAN EVALUATION ===")
        evaluation = self.human_operator.evaluate_ai_recommendation(
            recommendation, explanation, context, customer_constraints
        )
        self._display_human_evaluation(evaluation, recommendation)

        print("\n=== STEP 3: COLLABORATIVE DECISION ===")
        final_decision = self._make_collaborative_decision(
            recommendation, evaluation, explanation
        )
        print("\n=== STEP 4: FEEDBACK RECORDING ===")
        learning_outcomes = self._capture_learning(recommendation, evaluation, final_decision)

        decision = CollaborativeDecision(
            timestamp=context.timestamp,
            ai_recommendation=recommendation,
            ai_explanation=explanation,
            human_feedback=evaluation,
            human_insights=evaluation.get("insights", []),
            final_decision=final_decision,
            collaboration_quality=self._calculate_collaboration_quality(
                recommendation, evaluation, final_decision
            ),
            learning_outcomes=learning_outcomes,
        )
        self._display_final_result(decision)
        self.collaboration_history.append(decision)
        return decision

    def _display_ai_recommendation(
        self, recommendation: Dict[str, Any], explanation: AIExplanation
    ) -> None:
        """Present the proposal and its heuristic explanation to the operator."""
        print("\nAI RECOMMENDATION REPORT")
        print(f"Proposed moves: {len(recommendation['moves'])}")
        print(f"Time window: {recommendation['time_window']}")
        print(f"Heuristic completion estimate: {recommendation['expected_completion']:.1%}")
        print(f"Illustrative cost estimate: ${recommendation['estimated_cost']:,.0f}")
        print("\nREASONING")
        print(f"1. {explanation.primary_reasoning}")
        print(f"2. {explanation.secondary_reasoning}")
        print(f"3. Heuristic priority score: {recommendation['priority_score']:.2f}")
        print(f"\nHeuristic confidence score: {explanation.confidence_level:.2f} ({self._confidence_label(explanation.confidence_level)})")
        print("\nALTERNATIVES")
        for index, alternative in enumerate(explanation.alternatives, start=1):
            print(
                f"Option {index}: {alternative['description']} — "
                f"{len(alternative['moves'])} moves, ${alternative['cost']:,.0f}, "
                f"completion estimate {alternative['completion']:.1%}"
            )
        print("\nRISKS TO REVIEW")
        for risk in explanation.risks[:5]:
            print(f"- {risk}")
        print("\nILLUSTRATIVE OUTCOME INDICATORS")
        for key, value in explanation.expected_outcomes.items():
            print(f"- {key.replace('_', ' ').title()}: {value:.2f}")

    def _display_human_evaluation(
        self, evaluation: Dict[str, Any], recommendation: Dict[str, Any]
    ) -> None:
        """Show feedback, concerns, and any differences from the proposal."""
        print("\nHUMAN OPERATOR EVALUATION")
        print(f"Operator: {self.human_operator.name} ({self.human_operator.experience_years} years of experience)")
        print(f"Feedback type: {evaluation['feedback_type'].upper()}")
        print(f"Heuristic review confidence: {evaluation['confidence']:.2f}")
        if evaluation["feedback_type"] == "modify":
            modified = evaluation["modifications"]
            print(f"Proposed move count: {len(recommendation['moves'])}")
            print(f"Modified move count: {len(modified['moves'])}")
            print(f"Modified illustrative cost: ${modified['estimated_cost']:,.0f}")
            print(f"Modified time window: {modified['time_window']}")
        print(f"\nReasoning: {evaluation['reasoning']}")
        for concern in evaluation["concerns"]:
            print(f"- Concern: {concern}")
        for insight in evaluation["insights"]:
            print(f"- Recorded insight ({insight.insight_type}): {insight.reasoning}")

    def _make_collaborative_decision(
        self,
        ai_recommendation: Dict[str, Any],
        human_evaluation: Dict[str, Any],
        ai_explanation: AIExplanation,
    ) -> Dict[str, Any]:
        """Select the recommendation, a human modification, an alternative, or a cautious subset."""
        feedback = human_evaluation["feedback_type"]
        if feedback == "accept":
            final_decision = copy.deepcopy(ai_recommendation)
            decision_source = "ai_accepted"
        elif feedback == "modify":
            final_decision = copy.deepcopy(human_evaluation["modifications"])
            decision_source = "human_modified"
        elif feedback == "request_alternatives" and ai_explanation.alternatives:
            best_alternative = min(
                ai_explanation.alternatives, key=lambda alternative: alternative["cost"]
            )
            final_decision = {
                "moves": copy.deepcopy(best_alternative["moves"]),
                "time_window": best_alternative["time_window"],
                "estimated_cost": sum(move["cost"] for move in best_alternative["moves"]),
                "priority_score": ai_recommendation["priority_score"] * 0.8,
                "expected_completion": best_alternative["completion"],
                "source": "alternative_selected",
            }
            decision_source = "alternative_selected"
        else:
            moves = copy.deepcopy(ai_recommendation["moves"][:2])
            final_decision = {
                "moves": moves,
                "time_window": "2_hours",
                "estimated_cost": sum(move["cost"] for move in moves),
                "priority_score": ai_recommendation["priority_score"] * 0.6,
                "expected_completion": 0.4,
                "source": "conservative_plan",
            }
            decision_source = "conservative_plan"

        final_decision["decision_source"] = decision_source
        final_decision["human_confidence"] = human_evaluation["confidence"]
        final_decision["ai_confidence"] = ai_explanation.confidence_level
        assert final_decision["estimated_cost"] == sum(
            move["cost"] for move in final_decision["moves"]
        )
        return final_decision

    def _capture_learning(
        self,
        ai_recommendation: Dict[str, Any],
        human_evaluation: Dict[str, Any],
        final_decision: Dict[str, Any],
    ) -> List[str]:
        """Store human feedback without treating an unobserved outcome as proven."""
        outcomes = []
        for insight in human_evaluation.get("insights", []):
            outcomes.append(f"Recorded insight: {insight.reasoning}")
            self.learning_database.append(
                LearningCapture(
                    interaction_id=f"interaction_{len(self.learning_database) + 1}",
                    timestamp=datetime.now(),
                    situation_description=insight.context,
                    human_modification=copy.deepcopy(final_decision),
                    original_ai_recommendation=copy.deepcopy(ai_recommendation),
                    outcome_better=None,
                    learning_extracted=insight.reasoning,
                    generalizable_pattern=False,
                    confidence_improvement=0.0,
                )
            )
        if human_evaluation["feedback_type"] == "modify":
            outcomes.append("Recorded a human modification for later review.")
        return outcomes

    def _calculate_collaboration_quality(
        self,
        ai_recommendation: Dict[str, Any],
        human_evaluation: Dict[str, Any],
        final_decision: Dict[str, Any],
    ) -> float:
        """Calculate a transparent interaction score, not decision accuracy."""
        score = 0.5
        if human_evaluation["feedback_type"] == "accept":
            score += 0.2
        elif human_evaluation["feedback_type"] == "modify":
            score += 0.15
        if human_evaluation.get("insights"):
            score += 0.15
        if len(human_evaluation.get("reasoning", "")) > 50:
            score += 0.1
        return min(1.0, score)

    def _display_final_result(self, decision: CollaborativeDecision) -> None:
        """Summarize the final decision and the records created."""
        final = decision.final_decision
        print(f"\nFINAL DECISION: {final['decision_source'].upper()}")
        print(f"Moves in proposed plan: {len(final['moves'])}")
        print(f"Illustrative cost estimate: ${final['estimated_cost']:,.0f}")
        print(f"Interaction score: {decision.collaboration_quality:.1%}")
        print("The proposed moves have not been physically executed.")
        if decision.learning_outcomes:
            print("Feedback records:")
            for outcome in decision.learning_outcomes:
                print(f"- {outcome}")

    def _confidence_label(self, confidence: float) -> str:
        """Map the heuristic score to a qualitative display label."""
        if confidence >= 0.8:
            return "High"
        if confidence >= 0.6:
            return "Medium"
        if confidence >= 0.4:
            return "Low"
        return "Very Low"

    def analyze_collaboration_performance(self) -> Dict[str, Any]:
        """Summarize recorded sessions without inferring causal performance gains."""
        if not self.collaboration_history:
            return {"message": "No collaboration history is available."}
        decision_sources = [
            decision.final_decision.get("decision_source", "unknown")
            for decision in self.collaboration_history
        ]
        return {
            "total_sessions": len(self.collaboration_history),
            "avg_collaboration_quality": float(
                np.mean([d.collaboration_quality for d in self.collaboration_history])
            ),
            "decision_distribution": {
                source: decision_sources.count(source) / len(decision_sources)
                for source in sorted(set(decision_sources))
            },
            "learning_captured": len(self.learning_database),
            "interpretation": "Descriptive records only; no standalone baseline or outcome validation is modeled.",
        }

In [ ]:
def create_sample_scenario() -> Tuple[Dict[str, Any], OperationalContext, List[CustomerConstraint]]:
    """Create the demonstration yard, operating context, and customer inputs."""
    yard_blocks = {}
    for block_id in range(1, 13):
        current_load = int(np.random.randint(80, 130))
        yard_blocks[block_id] = YardBlock(
            id=block_id,
            current_load=current_load,
            capacity=120,
            utilization=current_load / 120,
            available_capacity=max(0, 120 - current_load),
        )

    timestamp = datetime.now().replace(hour=8, minute=15, second=0, microsecond=0)
    context = OperationalContext(
        timestamp=timestamp,
        weather_conditions={
            "temperature": 18.5,
            "wind_speed": 12.0,
            "visibility": 8.0,
            "precipitation": 0.0,
        },
        equipment_status={
            "crane_1": "operational",
            "crane_2": "operational",
            "crane_3": "maintenance",
            "crane_4": "operational",
            "crane_5": "operational",
            "crane_6": "operational",
        },
        staff_availability={"operators": 8, "supervisors": 2, "maintenance": 3},
        vessel_schedule=[
            {
                "name": "MSC Global Express",
                "eta": timestamp.replace(hour=14, minute=0),
                "priority": "high",
            }
        ],
        gate_traffic_level="medium",
        maintenance_activities=["crane_3_maintenance"],
        safety_constraints=["high_wind_protocol"],
        regulatory_requirements=["safety_first"],
    )
    customers = [
        CustomerConstraint(
            customer_id="CUST001",
            customer_name="Premium Shipping Co.",
            priority_level="premium",
            handling_requirements=["temperature_controlled", "priority_handling"],
            service_level_agreement={"max_delay": 30, "availability": 0.99},
            special_instructions=["minimal_handling", "protect_from_weather"],
            cost_sensitivity=0.3,
            time_sensitivity=0.9,
        ),
        CustomerConstraint(
            customer_id="CUST002",
            customer_name="Standard Logistics Ltd.",
            priority_level="standard",
            handling_requirements=["standard_handling"],
            service_level_agreement={"max_delay": 120, "availability": 0.95},
            special_instructions=[],
            cost_sensitivity=0.7,
            time_sensitivity=0.5,
        ),
    ]
    return {"blocks": yard_blocks}, context, customers


collaboration_system = HumanAICollaboration()
print("Human–AI housekeeping demonstration initialized.")
print(f"Operator: {collaboration_system.human_operator.name} ({collaboration_system.human_operator.experience_years} years of experience)")
print("Expertise areas: " + ", ".join(collaboration_system.human_operator.expertise_areas))

yard_state, context, customer_constraints = create_sample_scenario()
utilizations = [block.utilization for block in yard_state["blocks"].values()]
operational_cranes = sum(status == "operational" for status in context.equipment_status.values())
print(f"\nScenario time: {context.timestamp:%Y-%m-%d %H:%M}")
print(
    f"Weather inputs: {context.weather_conditions['wind_speed']:.1f} km/h wind, "
    f"{context.weather_conditions['temperature']:.1f}°C"
)
print(f"Equipment: {len(context.equipment_status)} cranes listed, {operational_cranes} operational")
print(f"Customer records: {len(customer_constraints)}")
print(f"Upcoming vessel: {context.vessel_schedule[0]['name']} at {context.vessel_schedule[0]['eta']:%H:%M}")
print(f"Mean block utilization: {np.mean(utilizations):.1%}")
print(f"Blocks above 90% utilization: {sum(value > 0.9 for value in utilizations)}")
print(f"Blocks below 70% utilization: {sum(value < 0.7 for value in utilizations)}")

In [ ]:
collaborative_decision = collaboration_system.simulate_collaborative_session(
    yard_state, context, customer_constraints
)

performance_analysis = collaboration_system.analyze_collaboration_performance()
print("\nSESSION RECORD SUMMARY")
for key, value in performance_analysis.items():
    if isinstance(value, dict):
        print(f"{key.replace('_', ' ').title()}:")
        for sub_key, sub_value in value.items():
            print(f"- {sub_key}: {sub_value:.1%}")
    else:
        print(f"{key.replace('_', ' ').title()}: {value}")

In [ ]:
def create_stress_scenario() -> Tuple[Dict[str, Any], OperationalContext, List[CustomerConstraint]]:
    """Create a higher-load scenario with adverse weather and constrained resources."""
    yard_blocks = {}
    for block_id in range(1, 13):
        current_load = int(np.random.randint(100, 140))
        yard_blocks[block_id] = YardBlock(
            id=block_id,
            current_load=current_load,
            capacity=120,
            utilization=current_load / 120,
            available_capacity=max(0, 120 - current_load),
        )

    timestamp = datetime.now().replace(hour=10, minute=30, second=0, microsecond=0)
    context = OperationalContext(
        timestamp=timestamp,
        weather_conditions={
            "temperature": 15.0,
            "wind_speed": 35.0,
            "visibility": 5.0,
            "precipitation": 8.0,
        },
        equipment_status={
            "crane_1": "operational",
            "crane_2": "failure",
            "crane_3": "maintenance",
            "crane_4": "operational",
            "crane_5": "operational",
            "crane_6": "operational",
        },
        staff_availability={"operators": 5, "supervisors": 1, "maintenance": 2},
        vessel_schedule=[
            {"name": "Urgent Vessel Alpha", "eta": timestamp.replace(hour=12), "priority": "urgent"},
            {"name": "Express Vessel Beta", "eta": timestamp.replace(hour=13, minute=30), "priority": "high"},
        ],
        gate_traffic_level="high",
        maintenance_activities=["crane_2_repair", "crane_3_maintenance"],
        safety_constraints=["high_wind_protocol", "heavy_rain_protocol"],
        regulatory_requirements=["safety_first", "weather_restrictions"],
    )
    customers = [
        CustomerConstraint(
            customer_id="CUST001",
            customer_name="Premium Shipping Co.",
            priority_level="premium",
            handling_requirements=["temperature_controlled", "priority_handling"],
            service_level_agreement={"max_delay": 15, "availability": 0.995},
            special_instructions=["minimal_handling", "protect_from_weather", "urgent_priority"],
            cost_sensitivity=0.2,
            time_sensitivity=0.95,
        ),
        CustomerConstraint(
            customer_id="CUST003",
            customer_name="VIP Global Logistics",
            priority_level="premium",
            handling_requirements=["vip_handling", "priority_processing"],
            service_level_agreement={"max_delay": 20, "availability": 0.99},
            special_instructions=["no_delays", "protect_from_weather"],
            cost_sensitivity=0.1,
            time_sensitivity=0.9,
        ),
        CustomerConstraint(
            customer_id="CUST004",
            customer_name="Standard Logistics Ltd.",
            priority_level="standard",
            handling_requirements=["standard_handling"],
            service_level_agreement={"max_delay": 180, "availability": 0.90},
            special_instructions=[],
            cost_sensitivity=0.8,
            time_sensitivity=0.4,
        ),
    ]
    return {"blocks": yard_blocks}, context, customers


print("MULTIPLE SCENARIO ANALYSIS")
stress_yard, stress_context, stress_customers = create_stress_scenario()
print(
    f"Stress weather inputs: {stress_context.weather_conditions['wind_speed']:.1f} km/h wind, "
    f"{stress_context.weather_conditions['precipitation']:.1f} mm/h precipitation"
)
print(
    f"Equipment: {len(stress_context.equipment_status)} cranes listed, "
    f"{sum(status == 'operational' for status in stress_context.equipment_status.values())} operational"
)
print(f"Available operators: {stress_context.staff_availability['operators']}")
print(f"Vessel records: {len(stress_context.vessel_schedule)}")
print(f"Customer records: {len(stress_customers)}")

stress_decision = collaboration_system.simulate_collaborative_session(
    stress_yard, stress_context, stress_customers
)

normal_moves = len(collaborative_decision.final_decision["moves"])
normal_cost = collaborative_decision.final_decision["estimated_cost"]
normal_quality = collaborative_decision.collaboration_quality
normal_learning = len(collaborative_decision.learning_outcomes)
stress_moves = len(stress_decision.final_decision["moves"])
stress_cost = stress_decision.final_decision["estimated_cost"]
stress_quality = stress_decision.collaboration_quality
stress_learning = len(stress_decision.learning_outcomes)

print("\nSCENARIO OUTPUT COMPARISON")
print(f"{'Scenario':<20} {'Moves':<10} {'Illustrative cost':<20} {'Interaction score':<20} {'Feedback records':<18}")
print("-" * 88)
print(f"{'Normal inputs':<20} {normal_moves:<10} ${normal_cost:<19,.0f} {normal_quality:<20.1%} {normal_learning:<18}")
print(f"{'Stress inputs':<20} {stress_moves:<10} ${stress_cost:<19,.0f} {stress_quality:<20.1%} {stress_learning:<18}")
print("These are outputs of the stated rules, not evidence of operational performance.")

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# The score is a rule-based interaction metric, so the plot does not measure decision accuracy.
session_numbers = list(range(1, len(collaboration_system.collaboration_history) + 1))
quality_scores = [decision.collaboration_quality for decision in collaboration_system.collaboration_history]
if quality_scores:
    ax1.plot(session_numbers, quality_scores, "bo-", linewidth=2, markersize=8, label="Interaction score")
    ax1.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, label="Reference line (80%)")
    ax1.set_xlabel("Session number")
    ax1.set_ylabel("Rule-based interaction score")
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "No session records", ha="center", va="center", transform=ax1.transAxes)
ax1.set_title("Recorded Interaction Scores")
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

sources = [decision.final_decision.get("decision_source", "unknown") for decision in collaboration_system.collaboration_history]
if sources:
    labels = sorted(set(sources))
    counts = [sources.count(label) for label in labels]
    ax2.pie(counts, labels=labels, autopct="%1.1f%%", startangle=90)
else:
    ax2.text(0.5, 0.5, "No decision records", ha="center", va="center", transform=ax2.transAxes)
ax2.set_title("Recorded Decision Sources")

learning_count = len(collaboration_system.learning_database)
if learning_count:
    ax3.plot(range(1, learning_count + 1), range(1, learning_count + 1), "g-o", linewidth=2)
else:
    ax3.text(0.5, 0.5, "No feedback records", ha="center", va="center", transform=ax3.transAxes)
ax3.set_xlabel("Feedback record number")
ax3.set_ylabel("Cumulative records stored")
ax3.set_title("Stored Feedback Records")
ax3.grid(True, alpha=0.3)

scenarios = ["Normal inputs", "Stress inputs"]
move_values = [normal_moves, stress_moves]
cost_values = [normal_cost, stress_cost]
quality_values = [normal_quality, stress_quality]
positions = np.arange(len(scenarios))
width = 0.25
max_moves = max(move_values) or 1
max_cost = max(cost_values) or 1
normalized_moves = [value / max_moves for value in move_values]
normalized_cost = [value / max_cost for value in cost_values]
bars_moves = ax4.bar(positions - width, normalized_moves, width, label="Move count (normalized)", color="blue", alpha=0.7)
bars_cost = ax4.bar(positions, normalized_cost, width, label="Cost estimate (normalized)", color="red", alpha=0.7)
bars_quality = ax4.bar(positions + width, quality_values, width, label="Interaction score", color="green", alpha=0.7)
ax4.set_xlabel("Generated scenario")
ax4.set_ylabel("Normalized value or score")
ax4.set_title("Descriptive Scenario Output Comparison")
ax4.set_xticks(positions)
ax4.set_xticklabels(scenarios)
ax4.set_ylim(0, 1.2)
ax4.legend()
ax4.grid(True, alpha=0.3)
for bars, values, formatter in [
    (bars_moves, move_values, lambda value: f"{value}"),
    (bars_cost, cost_values, lambda value: f"${value:.0f}"),
    (bars_quality, quality_values, lambda value: f"{value:.1%}"),
]:
    for bar, value in zip(bars, values):
        ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, formatter(value), ha="center", va="bottom", fontsize=9)

fig.suptitle("Human–AI Workflow: Descriptive Outputs", fontsize=15)
fig.tight_layout()
plt.show()

In [ ]:
final_analysis = collaboration_system.analyze_collaboration_performance()
print("HUMAN–AI WORKFLOW SUMMARY")
print(f"Recorded sessions: {final_analysis.get('total_sessions', 0)}")
print(f"Mean rule-based interaction score: {final_analysis.get('avg_collaboration_quality', 0):.1%}")
print(f"Stored feedback records: {final_analysis.get('learning_captured', 0)}")
print(final_analysis.get("interpretation", final_analysis.get("message", "")))

if "decision_distribution" in final_analysis:
    print("\nDecision-source proportions:")
    for source, proportion in final_analysis["decision_distribution"].items():
        print(f"- {source.replace('_', ' ').title()}: {proportion:.1%}")

print("\nWhat this demonstration illustrates:")
print("1. The AI recommendation is accompanied by its rule-based reasoning and uncertainty indicators.")
print("2. Human review can accept, modify, request an alternative, or reject a proposal.")
print("3. Human observations can be stored for later analysis; they do not automatically change the recommender.")
print("4. Scenario plots summarize generated outputs and do not compare against measured human-only or AI-only baselines.")
print("5. Real deployment would require validated feasibility, safety, cost, and outcome models.")

print("\nImplementation topics for further study:")
print("- Validate block-level assumptions against container- and equipment-level operational data.")
print("- Define auditable decision authority, safety review, and escalation procedures.")
print("- Assess recommendation quality against observed outcomes and appropriate baselines.")
print("- Establish governance for feedback retention, privacy, and any future model updates.")

## Summary and Key Insights

This notebook presents a human–AI housekeeping workflow in which a rule-based recommender creates a proposal, a human operator reviews it, and the system records the resulting decision and feedback.

### Technical structure

- **Recommendation:** block utilization, weather, equipment status, vessel timing, and customer priorities feed explicit rules.
- **Explanation:** the system displays the main reasoning, heuristic confidence, risks, alternatives, and illustrative outcome indicators.
- **Human review:** rule-based checks identify customer, safety, staffing, and operational concerns. The operator may accept, modify, request an alternative, or reject the proposal.
- **Decision record:** the selected proposal or modification is stored with the feedback and a transparent interaction score.
- **Scenario analysis:** normal and stress inputs illustrate how the same workflow responds to different conditions.

### How to interpret the simulation

The confidence and interaction scores are defined by code rules; neither is a probability of correctness or a validated measure of decision quality. Cost and completion values are illustrative estimates. The generated scenarios and their comparison do not demonstrate superiority over standalone human or AI decision-making. Feedback records preserve observations but do not train or update the recommender.

The central educational point is the division of roles: the AI component organizes data and presents a proposal, while human review provides a structured opportunity to consider context and constraints. A real terminal application would require validated operational models, safety and feasibility checks, outcome data, and appropriate oversight before recommendations could guide physical operations.